# MasaStore — Next-Item Recommendation (Day 11 Bonus)

RandomForest-based product recommender. Predicts the most likely next purchase for a user based on:
- Purchase history (counts of distinct orders containing each item)
- Favorites
- Current cart (TEMP order)
- Country (geo signal)
- Engagement stats (order count, avg basket size, total spent, recency)

Synthetic users are sampled along two axes — a **category persona** (Tech Enthusiast, Home Cook, Fashion Lover, Fitness Fan, Generalist) and a **spending persona** (Saver, Average, Spender) that weights items inside a category by `1 / price^alpha`. A 35% repeat-purchase bias creates the "habit" signal in `purchase_hist`.

**Output:** a single `recommender.joblib` file consumed by the FastAPI `/recommendation/` endpoint.

**How to run in Colab:**
1. Upload this notebook to https://colab.research.google.com
2. Runtime → Run all
3. The last cell downloads `recommender.joblib` to your machine

## 0. Install dependencies
Run once on Colab. If already installed locally, the cell is a no-op.

In [ ]:
!pip install -q scikit-learn pandas joblib

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timedelta

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

## 1. Item catalog

30 products across 5 categories. Each item has an `id`, `item_name`, `category`, `price` — matching the shape of our `item` table.

In [ ]:
# Prices match resources/db-migrations/init.sql.
CATEGORIES = {
    "Tech": [
        ("Headphones", 89.00), ("Laptop", 899.00), ("Smartwatch", 249.00),
        ("Bluetooth Speaker", 69.00), ("Tablet", 399.00), ("Wireless Mouse", 29.00),
    ],
    "Kitchen": [
        ("Coffee Maker", 149.00), ("Blender", 69.00), ("Air Fryer", 129.00),
        ("Knife Set", 99.00), ("Mixer", 179.00), ("Toaster", 39.00),
    ],
    "Fashion": [
        ("T-Shirt", 19.00), ("Jeans", 59.00), ("Sneakers", 89.00),
        ("Jacket", 149.00), ("Cap", 19.00), ("Belt", 39.00),
    ],
    "Home": [
        ("Lamp", 39.00), ("Pillow", 19.00), ("Bedsheet", 49.00),
        ("Candle", 14.00), ("Curtains", 79.00), ("Vase", 29.00),
    ],
    "Sports": [
        ("Yoga Mat", 39.00), ("Dumbbells", 79.00), ("Running Shoes", 99.00),
        ("Sports Bottle", 15.00), ("Backpack", 59.00), ("Resistance Band", 25.00),
    ],
}

rows = []
item_id = 1
for category, items_list in CATEGORIES.items():
    for name, price in items_list:
        rows.append({"id": item_id, "item_name": name, "category": category, "price": price})
        item_id += 1

items_df = pd.DataFrame(rows)
NUM_ITEMS = len(items_df)

item_id_to_index = {row.id: i for i, row in items_df.iterrows()}
index_to_item_id = {i: item_id for item_id, i in item_id_to_index.items()}
price_lookup = dict(zip(items_df.id, items_df.price))

print(f"Total items: {NUM_ITEMS}")
items_df

## 2. Personas

Five personas, each defined by a probability distribution over categories. Every synthetic user is assigned to one persona; their orders are drawn from that persona's distribution. This is what gives the model a *real* signal to learn — without distinct shopping patterns, recommendation is impossible.

In [ ]:
PERSONAS = {
    "Tech Enthusiast": {"Tech": 0.70, "Home": 0.15, "Sports": 0.10, "Kitchen": 0.03, "Fashion": 0.02},
    "Home Cook":       {"Kitchen": 0.65, "Home": 0.20, "Tech": 0.08, "Fashion": 0.05, "Sports": 0.02},
    "Fashion Lover":   {"Fashion": 0.70, "Home": 0.15, "Sports": 0.10, "Tech": 0.03, "Kitchen": 0.02},
    "Fitness Fan":     {"Sports": 0.65, "Fashion": 0.15, "Home": 0.10, "Tech": 0.05, "Kitchen": 0.05},
    "Generalist":      {"Tech": 0.20, "Kitchen": 0.20, "Fashion": 0.20, "Home": 0.20, "Sports": 0.20},
}

# alpha = price-sensitivity exponent inside a category: weight ∝ 1 / price^alpha.
# alpha=0 → uniform within the category; alpha=1 → strong cheap-bias.
SPENDING_PERSONAS = {
    "Saver":   1.0,
    "Average": 0.5,
    "Spender": 0.0,
}

def persona_item_distribution(category_persona, spending_persona):
    """Probability vector over all NUM_ITEMS items for this combined persona.
    Step 1: distribute total probability across categories per PERSONAS.
    Step 2: within each category, weight items by 1/price^alpha per SPENDING_PERSONAS.
    """
    dist = np.zeros(NUM_ITEMS)
    cat_probs = PERSONAS[category_persona]
    alpha = SPENDING_PERSONAS[spending_persona]
    for cat, p in cat_probs.items():
        cat_items = items_df[items_df.category == cat]
        prices = cat_items.price.values
        if alpha > 0:
            weights = 1.0 / (prices ** alpha)
        else:
            weights = np.ones(len(cat_items))
        weights = weights / weights.sum()
        for w, (_, item) in zip(weights, cat_items.iterrows()):
            dist[item_id_to_index[item.id]] = p * w
    return dist / dist.sum()

# Sanity check: top item per (category persona, Saver vs Spender).
for cp in PERSONAS:
    d_saver = persona_item_distribution(cp, "Saver")
    d_spender = persona_item_distribution(cp, "Spender")
    top_saver = items_df.iloc[d_saver.argmax()]
    top_spender = items_df.iloc[d_spender.argmax()]
    print(f"{cp:18s}  Saver→{top_saver.item_name:18s} (${top_saver.price:>6.2f})   "
          f"Spender→{top_spender.item_name:18s} (${top_spender.price:>6.2f})")

## 3. Generate synthetic users + orders + favorites + carts

In [ ]:
COUNTRIES = ["IL", "US", "UK", "DE", "FR", "ES", "IT", "JP", "BR", "IN"]
NUM_USERS = 5000
NOW = datetime(2026, 5, 25)
REPEAT_BIAS = 0.35  # probability that a basket pick repeats a prior purchase

users = []
all_orders = []
all_favorites = []

for uid in range(1, NUM_USERS + 1):
    category_persona = random.choice(list(PERSONAS.keys()))
    spending_persona = random.choice(list(SPENDING_PERSONAS.keys()))
    country = random.choice(COUNTRIES)
    users.append({
        "id": uid,
        "country": country,
        "category_persona": category_persona,
        "spending_persona": spending_persona,
    })

    dist = persona_item_distribution(category_persona, spending_persona)
    bought_so_far: list = []

    n_orders = random.randint(3, 15)
    for _ in range(n_orders):
        order_time = NOW - timedelta(days=random.randint(1, 365), seconds=random.randint(0, 86399))
        basket_size = random.randint(1, 4)
        for _ in range(basket_size):
            if bought_so_far and random.random() < REPEAT_BIAS:
                idx = random.choice(bought_so_far)
            else:
                idx = np.random.choice(NUM_ITEMS, p=dist)
            bought_so_far.append(idx)
            all_orders.append({
                "user_id": uid,
                "item_id": index_to_item_id[idx],
                "quantity": random.randint(1, 3),
                "timestamp": order_time,
                "is_temp": False,
            })

    # TEMP cart picks are fresh intents — no repeat bias here.
    for _ in range(random.randint(0, 3)):
        idx = np.random.choice(NUM_ITEMS, p=dist)
        all_orders.append({
            "user_id": uid,
            "item_id": index_to_item_id[idx],
            "quantity": 1,
            "timestamp": NOW,
            "is_temp": True,
        })

    n_fav = random.randint(1, 5)
    fav_indices = np.random.choice(NUM_ITEMS, size=n_fav, replace=False, p=dist)
    for idx in fav_indices:
        all_favorites.append({"user_id": uid, "item_id": index_to_item_id[idx]})

users_df = pd.DataFrame(users)
orders_df = pd.DataFrame(all_orders)
favorites_df = pd.DataFrame(all_favorites).drop_duplicates(subset=["user_id", "item_id"])

print(f"Users: {len(users_df)}")
print(f"Completed order line-items: {(~orders_df.is_temp).sum()}")
print(f"TEMP cart line-items: {orders_df.is_temp.sum()}")
print(f"Favorites: {len(favorites_df)}")
print()
print("Category persona distribution:")
print(users_df.category_persona.value_counts())
print()
print("Spending persona distribution:")
print(users_df.spending_persona.value_counts())

## 4. Feature engineering

For each user we build one feature vector + label using **leave-one-out**:
the most recent completed purchase becomes the label; everything before it becomes the features.

**Feature vector layout:**
1. Country one-hot (10)
2. Stats: order_count, avg_basket_size, total_spent, days_since_last_order (4)
3. Purchase histogram per item (30)
4. Favorites binary (30)
5. Current TEMP cart binary (30)

In [ ]:
def build_user_features(user_id, exclude_last=True):
    """Returns (feature_vector, label_item_id). label is None if no completed orders.

    purchase_hist counts DISTINCT ORDERS containing the item, not total
    quantity. Buying 3 of an item in one order counts as 1; buying it in 3
    separate orders counts as 3. total_spent still uses real quantity.
    """
    user = users_df[users_df.id == user_id].iloc[0]
    completed = orders_df[(orders_df.user_id == user_id) & (~orders_df.is_temp)].sort_values("timestamp")
    temp_cart = orders_df[(orders_df.user_id == user_id) & (orders_df.is_temp)]
    favs = favorites_df[favorites_df.user_id == user_id]

    label_item_id = None
    if exclude_last and len(completed) > 0:
        label_item_id = completed.iloc[-1].item_id
        completed = completed.iloc[:-1]

    purchase_hist = np.zeros(NUM_ITEMS)
    for _, group in completed.groupby("timestamp"):
        for item_id in group.item_id.unique():
            purchase_hist[item_id_to_index[item_id]] += 1

    fav_vec = np.zeros(NUM_ITEMS)
    for _, row in favs.iterrows():
        fav_vec[item_id_to_index[row.item_id]] = 1

    cart_vec = np.zeros(NUM_ITEMS)
    for _, row in temp_cart.iterrows():
        cart_vec[item_id_to_index[row.item_id]] = 1

    orders_grouped = completed.groupby("timestamp")
    order_count = orders_grouped.ngroups
    avg_basket_size = orders_grouped.size().mean() if order_count else 0.0
    total_spent = float(sum(price_lookup[r.item_id] * r.quantity for _, r in completed.iterrows()))
    days_since = (NOW - completed.timestamp.max()).days if order_count else 365

    country_vec = np.zeros(len(COUNTRIES))
    country_vec[COUNTRIES.index(user.country)] = 1

    feature = np.concatenate([
        country_vec,
        [order_count, avg_basket_size, total_spent, days_since],
        purchase_hist,
        fav_vec,
        cart_vec,
    ])
    return feature, label_item_id

X_list, y_list = [], []
for uid in users_df.id:
    feat, label = build_user_features(uid, exclude_last=True)
    if label is not None:
        X_list.append(feat)
        y_list.append(item_id_to_index[label])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)

feature_names = (
    [f"country_{c}" for c in COUNTRIES] +
    ["order_count", "avg_basket_size", "total_spent", "days_since_last_order"] +
    [f"purchase_hist_{i}" for i in range(NUM_ITEMS)] +
    [f"fav_{i}" for i in range(NUM_ITEMS)] +
    [f"cart_{i}" for i in range(NUM_ITEMS)]
)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Feature dimensions: {len(feature_names)}")

## 5. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")

## 6. Build the model

`RandomForestClassifier` from scikit-learn — chosen over a Keras ANN for three reasons:
- It is part of the course material (Supervised Learning), unlike Keras/scikeras
- It needs no GPU, no version pinning, no extra dependencies — and avoids the `__sklearn_tags__` incompatibility between scikit-learn 1.6+ and scikeras
- It exposes `feature_importances_`, giving interpretable insight into what drives the recommendations

**No StandardScaler:** RandomForest splits on threshold values, so feature scaling has no effect on its decisions. We dropped the scaler to keep the Pipeline honest.

**Pipeline:** single step `(classifier: RandomForestClassifier(...))`. We kept the Pipeline wrapper so the same `joblib.load(...)['pipeline'].predict_proba(...)` call works on Day 12 unchanged.

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    random_state=101,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("classifier", clf),
])

## 7. Train

In [ ]:
pipeline.fit(X_train, y_train)

## 8. Evaluate

With 30 items, random baseline is 1/30 ≈ 3.3%. We expect Top-1 ≈ 25-45% and Top-3 ≈ 55-75% — the personas create learnable patterns but not noise-free ones.

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

top1 = accuracy_score(y_test, y_pred)
top3 = top_k_accuracy_score(y_test, y_proba, k=3, labels=np.arange(NUM_ITEMS))
top5 = top_k_accuracy_score(y_test, y_proba, k=5, labels=np.arange(NUM_ITEMS))

print(f"Top-1 accuracy: {top1:.3f}")
print(f"Top-3 accuracy: {top3:.3f}")
print(f"Top-5 accuracy: {top5:.3f}")
print(f"Random baseline: {1/NUM_ITEMS:.3f}")

In [ ]:
# Category-level accuracy: did we at least recommend from the right category?
cat_correct = Counter()
cat_total = Counter()
for true_idx, pred_idx in zip(y_test, y_pred):
    true_cat = items_df.iloc[true_idx].category
    pred_cat = items_df.iloc[pred_idx].category
    cat_total[true_cat] += 1
    if true_cat == pred_cat:
        cat_correct[true_cat] += 1

print("Category-level accuracy:")
for cat in CATEGORIES:
    n = cat_total[cat]
    if n > 0:
        pct = cat_correct[cat] / n
        print(f"  {cat:10s}: {pct:.2%}  ({cat_correct[cat]}/{n})")

## 8.5 Feature importances

RandomForest gives us interpretability for free: `feature_importances_` tells us how much each input dimension contributed to the model's decisions, averaged across the forest. This is the kind of insight that makes the model defensible in the project review — we can point to *which signals* drove each recommendation.

In [ ]:
classifier = pipeline.named_steps["classifier"]
importances = classifier.feature_importances_

top_idx = np.argsort(importances)[::-1][:10]
print("Top 10 most important features:")
print()
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {feature_names[idx]:30s}  importance = {importances[idx]:.4f}")

## 9. Export

Single `recommender.joblib` containing the fitted pipeline + all metadata FastAPI needs to make predictions on Day 12.

In [ ]:
bundle = {
    "pipeline": pipeline,
    "items": items_df.to_dict(orient="records"),
    "item_id_to_index": item_id_to_index,
    "index_to_item_id": index_to_item_id,
    "feature_names": feature_names,
    "countries": COUNTRIES,
    "num_items": NUM_ITEMS,
    "version": "2.0",
    "trained_at": datetime.utcnow().isoformat(),
    "metrics": {"top1": float(top1), "top3": float(top3), "top5": float(top5)},
}
joblib.dump(bundle, "recommender.joblib")
size_kb = os.path.getsize("recommender.joblib") / 1024
print(f"Saved recommender.joblib ({size_kb:.1f} KB)")

## 10. Demo prediction

Pick a random test user and show their actual next purchase next to the model's top-3 recommendations.

In [ ]:
for sample_idx in range(5):
    sample_feature = X_test[sample_idx].reshape(1, -1)
    sample_true = y_test[sample_idx]

    probs = pipeline.predict_proba(sample_feature)[0]
    top3_indices = np.argsort(probs)[::-1][:3]

    true_item = items_df.iloc[sample_true]
    print(f"--- Test user #{sample_idx} ---")
    print(f"Actual next purchase: {true_item.item_name} ({true_item.category})")
    print("Top-3 recommendations:")
    for rank, idx in enumerate(top3_indices, 1):
        item = items_df.iloc[idx]
        marker = " ✓" if idx == sample_true else ""
        print(f"  {rank}. {item.item_name:20s} ({item.category:10s})  prob={probs[idx]:.3f}{marker}")
    print()

## 11. Download the artifact

On Colab this triggers a browser download. Put the file at `ml/artifacts/recommender.joblib` in the project repo for Day 12.

In [ ]:
try:
    from google.colab import files
    files.download("recommender.joblib")
except ImportError:
    print("Not in Colab — recommender.joblib is in the current working directory.")